## Preliminares

In [1]:
# Importar modulos y funciones necesarias
import pandas as pd
import numpy as np
import os
from datetime import datetime
from sklearn.model_selection import cross_val_score, GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import plotly.express as px
from src.evaluators import *

In [2]:
# Abrir archivo clean_data
data_folder = "data"
df = pd.read_parquet(f"{data_folder}/clean_data.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17986 entries, 0 to 17985
Data columns (total 44 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   Date                            17986 non-null  object  
 1   Ticker                          17986 non-null  object  
 2   Sector                          17986 non-null  category
 3   YearsSinceAdded                 17986 non-null  float64 
 4   Monthly_Return                  17986 non-null  float64 
 5   Monthly_Variance                17986 non-null  float64 
 6   Market_Covariance               17986 non-null  float64 
 7   operatingMargins                17986 non-null  float64 
 8   profitMargins                   17986 non-null  float64 
 9   ReturnOnAssets                  17986 non-null  float64 
 10  Revenue_YoY                     17986 non-null  float64 
 11  Revenue_QoQ                     17986 non-null  float64 
 12  EBITDA_YoY        

## Feature Engineering

Sección para crear variables en la fase de modelado. 
La mayor parte de las variables fue creada en la fase de extracción.

In [3]:
# Calcular las aceleraciones netas (Momento - Tendencia) para variables de crecimiento.
# Una vez calculadas, se eliminan las variables trimestrales (reemplazar=True)
variables_de_crecimiento = ['CPI_QoQ', 'M2_QoQ', 'Revenue_QoQ', 'EBITDA_QoQ', 'FCF_QoQ', 'Capex_QoQ']
df = calcular_acceleration_features(df, variables_de_crecimiento, reemplazar= True)

In [4]:
# Sectores poco significativos: 
# Solo InformationTechnology resulta relevante, y Energy marginalmente.
# se agrupan en la categoria "Other" el resto de los sectores
sectores_importantes = ['InformationTechnology', 'Energy']

df['Sector'] = np.where(df['Sector'].isin(sectores_importantes), df['Sector'], 'Other')

# Se vuelve a convertir en category
df['Sector'] = df['Sector'].astype('category')

In [5]:
# Se eliminan columnas con significancia menor a 0.01
df = df.drop(columns=[
    'CPI_YoY',
    'CPI_Acceleration',
    'Monthly_Return',
    'UnemploymentRate',
    '10Y2YSpread',
    'HighYieldSpread',
    'FedFundsRate',
    'M2_Acceleration',
    'M2_YoY',
    'Market_Covariance'
])

# Modelo de ensamblado de árboles RandomForest

In [6]:
# Se asegura el ordenamiento por fecha
df.sort_values(by='Date', inplace=True)

# Eliminar predictores
predictores_a_eliminar = ['PriceToBook_Transformed', 
                          'PE_Trailing_Transformed',
                          'EnterpriseToEbitda_Transformed', 
                          'Close_log',
                          'MarketCap_log',
                          'EnterpriseValue_log',
                          'TotalEquity',
                          'TotalAssets',
                          'TotalRevenue',
                          'Ticker',
                          'Date'
                          ]

# Se define la variable objetivo y las variables predictoras
label = 'Close_log'
X = df.drop(columns=predictores_a_eliminar) 
y = df[label]

# Columnas numéricas: 
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

# Variables categóricas:
categorical_cols = ['Sector']

# preprocesador: escala numéricas y codifica categóricas
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

pipe = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=10,
        max_features='sqrt',
        max_samples= 0.8,
        min_samples_split= 10         
        ))
])

print("Entrenando el modelo con datos completos...")
pipe.fit(X, y)
r2_completo = pipe.score(X, y)
print(f"Entrenamiento finalizado. R2 en datos completos: {r2_completo:.4f}")

Entrenando el modelo con datos completos...
Entrenamiento finalizado. R2 en datos completos: 0.7828


In [7]:
# Test de validación cruzada
# Configurar el validador de series temporales
tscv = TimeSeriesSplit(n_splits=5) # n_splits=5 creará 5 particiones temporales secuenciales

# 3. Test de validación cruzada temporal
cross_val_scores = cross_val_score(
    estimator=pipe, 
    X=X, 
    y=y, 
    cv=tscv,         
    scoring='r2',
    n_jobs=-1        
)

print(f"R² promedio Time Series CV: {cross_val_scores.mean():.4f} ± {cross_val_scores.std():.4f}")

R² promedio Time Series CV: 0.5726 ± 0.0448


In [8]:
# Importancia de factores en el modelo
rf_model = pipe.named_steps['model']
importances = rf_model.feature_importances_

# Obtener los nombres de las características después del preprocesamiento
preprocessor = pipe.named_steps['pre']
feature_names = preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
feature_importance_df.head(20)

,feature,importance
4,num__ReturnOnAssets,0.140186
9,num__NetDebt_to_EBITDA,0.077423
12,num__returnOnEquity_Transformed,0.065913
11,num__Capex_to_Revenue,0.059596
13,num__debtToEquity_Transformed,0.056780
3,num__profitMargins,0.055307
14,num__RelativeAssets_log,0.053697
17,num__currentRatio_log,0.051986
2,num__operatingMargins,0.050294
10,num__FCF_to_EBITDA,0.045286


## Aplicacion del modelo

Se generan 2 clusters segun las predicciones:
* Positive_Bias: si los residuos son mayores o iguales a cero.
* Negative_Bias: si los residuos son menores a cero.

In [9]:
# Se dividen los datos para predecir el valor de la última fecha disponible para cada ticker en el conjunto de test
X_train = X.iloc[:-len(df['Ticker'].unique())]  # Todos menos la última fecha de cada ticker
y_train = y.iloc[:-len(df['Ticker'].unique())]
X_test = X.iloc[-len(df['Ticker'].unique()):]   # Solo la última fecha de cada ticker
y_test = y.iloc[-len(df['Ticker'].unique()):]

# Recuperar el ticker usando el indice de y_test
tickers_test = df.loc[y_test.index, 'Ticker']

# Predicciones en el conjunto de test
y_pred = pipe.predict(X_test)

# Consolidar resultados individuales en un DataFrame
resultados_df = pd.DataFrame({
    'Ticker': tickers_test,
    'Predicho': y_pred,
    'Real': y_test
})

# Calcular el residuo para cada predicción
resultados_df['Residuo'] = resultados_df['Predicho'] - resultados_df['Real']

# Agrupar por ticker
resultados_agrupados = resultados_df.groupby('Ticker')[['Predicho', 'Real', 'Residuo']].mean()

# Generar el Cluster sobre el promedio de los residuos
resultados_agrupados['Cluster'] = ['Positive_Bias' if r >= 0 else 'Negative_Bias' 
                                   for r in resultados_agrupados['Residuo']]

# Visualizar
fig = px.scatter(
    resultados_agrupados, 
    x='Real', 
    y='Predicho', 
    color='Cluster',
    hover_name=resultados_agrupados.index, 
    labels={'Real':'Valor Real Promedio', 'Predicho':'Predicción Promedio', 'Cluster':'Sesgo del Modelo'},
    title='Predicciones vs Reales (Agrupado por Ticker)'
)

# Línea de identidad perfecta
min_val = resultados_agrupados['Real'].min()
max_val = resultados_agrupados['Real'].max()
fig.add_shape(type='line', x0=min_val, y0=min_val, x1=max_val, y1=max_val,
              line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster a nivel Ticker
over_mask = resultados_agrupados['Cluster'] == 'Positive_Bias'
under_mask = resultados_agrupados['Cluster'] == 'Negative_Bias'

print("\nEstadísticas por cluster (a nivel de Ticker):")
print(f"Overprediction: {over_mask.sum()} tickers, residuo medio global: {resultados_agrupados.loc[over_mask, 'Residuo'].mean():.4f}")
print(f"Underprediction: {under_mask.sum()} tickers, residuo medio global: {resultados_agrupados.loc[under_mask, 'Residuo'].mean():.4f}")


Estadísticas por cluster (a nivel de Ticker):
Overprediction: 198 tickers, residuo medio global: 0.4224
Underprediction: 255 tickers, residuo medio global: -0.4880


In [10]:
# Ordenar resultados por residuos
resultados_agrupados = resultados_agrupados.sort_values(by='Residuo', ascending=False)
# Positive Biases: Sobrepredicciones seleccionando solo valores positivos para los ratios Reales y Predichos
positive_biases = resultados_agrupados[(resultados_agrupados['Real'] > 0) & (resultados_agrupados['Predicho'] > 0) & (resultados_agrupados['Cluster'] == 'Positive_Bias')]

positive_biases.head(20)

,Predicho,Real,Residuo,Cluster
Ticker,,,,
TTD,4.731363,3.017009,1.714353,Positive_Bias
GEN,4.892300,3.248046,1.644254,Positive_Bias
CMCSA,4.693429,3.230804,1.462624,Positive_Bias
KVUE,4.420886,2.959328,1.461558,Positive_Bias
BSX,5.178884,3.870472,1.308413,Positive_Bias
CMG,4.787253,3.518536,1.268718,Positive_Bias
T,4.430124,3.194173,1.235951,Positive_Bias
TSCO,4.675290,3.462606,1.212684,Positive_Bias
CAG,3.900095,2.696315,1.203780,Positive_Bias


In [11]:
# Establer Ticker como columna para exportar resultados
positive_biases.reset_index(inplace=True)
positive_biases.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Ticker    198 non-null    object 
 1   Predicho  198 non-null    float64
 2   Real      198 non-null    float64
 3   Residuo   198 non-null    float64
 4   Cluster   198 non-null    object 
dtypes: float64(3), object(2)
memory usage: 7.9+ KB


In [12]:
# Se exportan los sesgos positivos a un archivo CSV para su análisis posterior 
# Guardar dataframe con cluster con fecha en el nombre del archivo

dia = datetime.now().day
mes = datetime.now().month
year = datetime.now().year

os.makedirs(f"{data_folder}/results", exist_ok=True) # Crear carpeta si no existe

positive_biases.to_csv(f"{data_folder}/results/positive_biases_{year}_{mes}_{dia}.csv", index=False)

## Anexo: optimización de hiperparámetros

In [13]:
ejecutar_celda = False

if ejecutar_celda:
    nombre_modelo = "Random Forest"
    print(f"Configurando GridSearchCV para {nombre_modelo}")

    # Pipeline usando el preprocesador específico para Random Forest
    modelo_base = Pipeline(steps=[
        ('preprocesador', preprocessor),
        ('rf', RandomForestRegressor(random_state=42))
    ])

    param_grid = {
        'rf__n_estimators': [300],
        'rf__max_depth': [10],
        'rf__min_samples_leaf': [10],
        'rf__min_samples_split': [10],
        'rf__max_samples': [0.8],
        'rf__max_features': ['sqrt']
    }

    # Configurar el GridSearchCV
    grid_search = GridSearchCV(
        estimator=modelo_base,
        param_grid=param_grid,
        scoring='r2',
        cv=tscv,
        n_jobs=-1,
        verbose=2
    )

    # Entrenar con datos completos
    print(f"Iniciando búsqueda de hiperparámetros. Esto tomará unos minutos.")
    grid_search.fit(X, y)

    # Resultados
    print("\n--- Búsqueda Finalizada ---")
    print("Mejores hiperparámetros encontrados:")
    print(grid_search.best_params_)